# Test final: generación aislada de los golds

Ejecutar **solo después de congelar y entrenar el adapter final**.
Se usa exactamente el formato de entrada del baseline: mensaje user, FP16 y
1200 tokens máximos, sin muestreo. QLoRA modifica únicamente los pesos mediante
el adapter; la inferencia vuelve a FP16 para coincidir con el baseline.

Archivos requeridos en Drive: `codex_missing_assets/benchmark_30_generation_only.jsonl`,
`results/baseline/caso_entrega1_generation_only.jsonl`, adapter y manifest final.
No se carga ningún archivo con referencias ni requisitos.


In [ ]:
%pip -q install "transformers==4.51.3" "peft==0.15.2" "accelerate==1.6.0"


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, time, gc, contextlib, subprocess
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
from peft import PeftModel

PACKAGE=Path('/content/drive/MyDrive/IA/entrega2_qlora')
PROJECT=Path('/content/drive/MyDrive/IA/qwen25_qlora_or_v2')
FINAL_RUN=PROJECT/'final/user_only_run01'
OUT=FINAL_RUN/'benchmark'
OUT.mkdir(exist_ok=True)
manifest=json.loads((FINAL_RUN/'manifest.json').read_text())
completed=json.loads((FINAL_RUN/'completed.json').read_text())
frozen=json.loads((PROJECT/'frozen_config.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert completed['manifest_sha256']==sha(FINAL_RUN/'manifest.json')
assert manifest['mode']=='final' and manifest['epochs']==frozen['epochs']
for filename,digest in completed['adapter_hashes'].items():
    assert sha(FINAL_RUN/'final_adapter'/filename)==digest
def read_jsonl(p):
    return [json.loads(s) for s in p.read_text(encoding='utf-8').splitlines() if s.strip()]
items=read_jsonl(PACKAGE/'codex_missing_assets/benchmark_30_generation_only.jsonl')
assert len(items)==len({x['id'] for x in items})==30
items+=read_jsonl(PACKAGE/'results/baseline/caso_entrega1_generation_only.jsonl')
assert len(items)==len({x['id'] for x in items})==31
for item in items:
    assert set(item)<={'id','title','problem','model_class','difficulty'}, 'Archivo de generación con campos extra'
assert torch.cuda.is_available()
set_seed(42)
tokenizer=AutoTokenizer.from_pretrained(manifest['model_id'],revision=manifest['revision'])
if tokenizer.pad_token is None: tokenizer.pad_token=tokenizer.eos_token
base=AutoModelForCausalLM.from_pretrained(manifest['model_id'],revision=manifest['revision'],
                                       torch_dtype=torch.float16,device_map={'':0})
model=PeftModel.from_pretrained(base,FINAL_RUN/'final_adapter',is_trainable=False)
model.eval()
model.config.use_cache=True
generation_manifest={'variant':'qlora_final','messages':'user_only','max_new_tokens':1200,
    'do_sample':False,'dtype':'float16','model_revision':manifest['revision'],
    'adapter_manifest_sha256':sha(FINAL_RUN/'completed.json'),
    'inputs_sha256':sha(PACKAGE/'codex_missing_assets/benchmark_30_generation_only.jsonl'),
    'diagnostic_sha256':sha(PACKAGE/'results/baseline/caso_entrega1_generation_only.jsonl'),
    'gpu':torch.cuda.get_device_name(0),'generation_config':model.generation_config.to_dict()}
manifest_file=OUT/'generation_manifest.json'
if manifest_file.exists():
    assert json.loads(manifest_file.read_text())==generation_manifest
else:
    manifest_file.write_text(json.dumps(generation_manifest,indent=2),encoding='utf-8')


In [ ]:
def generate_direct(problem: str, max_new_tokens: int = 1200):
    """Baseline limpio: mismo esquema de Direct Prompting, pero decodificación determinista."""
    messages = [{"role": "user", "content": problem}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    t0 = time.perf_counter()
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.perf_counter() - t0

    generated = out[0][inputs["input_ids"].shape[1]:]
    text = tokenizer.decode(generated, skip_special_tokens=True).strip()
    return {
        "candidate": text,
        "prompt_tokens": int(inputs["input_ids"].shape[1]),
        "output_tokens": int(generated.shape[0]),
        "latency_sec": latency,
    }


In [ ]:
destination=OUT/'final_generations_31.jsonl'
outputs=read_jsonl(destination) if destination.exists() else []
assert len(outputs)==len({r['id'] for r in outputs})
by_id={r['id']:r for r in outputs}
assert set(by_id)<={item['id'] for item in items}
for item in items:
    if item['id'] in by_id: continue
    result=generate_direct(item['problem'])
    result.update(id=item['id'],title=item['title'],split='benchmark' if item['id'].startswith('opt_') else 'diagnostic',
                  variant='qlora_final',model=manifest['model_id'],problem_sha256=hashlib.sha256(item['problem'].encode()).hexdigest(),
                  reached_token_budget=result['output_tokens']>=1200)
    # An exhausted budget is a flag, not proof that EOS was absent.
    with destination.open('a',encoding='utf-8') as f: f.write(json.dumps(result,ensure_ascii=False)+'\n')
    by_id[item['id']]=result
    print(item['id'],result['output_tokens'],'tokens',round(result['latency_sec'],1),'s')
assert len(by_id)==31
print('31 salidas persistidas. Ahora ejecutar el juez en una sesión limpia.')


## Demostración reproducible (opcional, después del test)

La entrada diagnóstica fue fijada desde la Entrega 1, no elegida tras ver la mejora.
La celda siguiente ejecuta base y adapter sobre esa misma entrada. La regeneración
base se informa como nueva ejecución; no sobrescribe el baseline histórico.


In [ ]:
RUN_LIVE_DEMO=False
if RUN_LIVE_DEMO:
    diagnostic=next(x for x in items if x['id']=='diagnostic_oftalmo')
    with model.disable_adapter():
        base_demo=generate_direct(diagnostic['problem'])
    adapted_demo=generate_direct(diagnostic['problem'])
    from IPython.display import display, Markdown
    display(Markdown('### Baseline directo (ejecución nueva)\n'+base_demo['candidate']))
    display(Markdown('### QLoRA final (misma entrada)\n'+adapted_demo['candidate']))
    (OUT/'live_demo.json').write_text(json.dumps({'id':diagnostic['id'],'baseline_new_run':base_demo,
        'qlora':adapted_demo},ensure_ascii=False,indent=2),encoding='utf-8')
